In [ ]:
import matplotlib.cm as cm
from matplotlib.colors import hsv_to_rgb, Normalize
import matplotlib.pyplot as plt
import imageio
import numpy as np
import torch
from depth_from_events.dsec import DsecSequence

In [ ]:
def event_frame_to_rgb(event_frame, saturation_percentile=0.8, gain=1.0):
    # convert 2ch event frame (positive/negative event counts) to RGB image
    
    rgb = torch.stack((event_frame[0], torch.zeros_like(event_frame[0]), event_frame[1]), dim=0)
    # get the 80th percentile of the positive and negative event counts
    frame_max = max(torch.quantile(event_frame[0], saturation_percentile), torch.quantile(event_frame[1], saturation_percentile))
    rgb = (gain * rgb / frame_max * 255.).numpy().transpose(1, 2, 0).astype(np.uint8)
    return rgb

In [ ]:
def event_frame_to_image(frame, pol_channels=[0, 1]):
    """
    Convert an event frame to an RGB image, where the red channel
    represents negative events and the blue channel positive events.

    Args:
        frame (np.ndarray): Event frame with shape (?, height, width).
        pol_channels (list): Indices of the negative and positive polarity channels.

    Returns:
        np.ndarray: RGB image of the event frame.
    """

    # event frame (c, h, w), with channels neg and pos polarity
    _, h, w = frame.shape
    frame = frame[pol_channels]

    # normalize per channel
    almost_max = np.percentile(frame, 99, axis=(1, 2), keepdims=True)
    almost_min = np.percentile(frame, 1, axis=(1, 2), keepdims=True)
    if (almost_min != almost_max).all():
        frame_norm = (frame - almost_min) / (almost_max - almost_min)
    else:
        frame_norm = frame
    frame_norm = np.clip(frame_norm, 0, 1)

    # write to rgb image of ints
    frame_rgb = np.zeros((h, w, 3), dtype=np.uint8)
    neg, pos = frame_norm
    frame_rgb[:, :, 0] = neg * 255  # red
    frame_rgb[:, :, 2] = pos * 255  # blue

    return frame_rgb


In [ ]:
def disparity_map_to_image(disparity, mapper=None):
    """
    Convert a disparity map to an RGB image.
    Source: https://github.com/uzh-rpg/DSEC/blob/main/scripts/dataset/visualization.py

    Args:
        disparity (np.ndarray): Disparity map with shape (1, height, width).

    Returns:
        np.ndarray: RGB image of the disparity map.
    """

    # check shape
    assert disparity.ndim == 3 and disparity.shape[
        0] == 1, "Disparity must have shape (1, height, width)."

    # disparity magnitude for nonzero pixels
    disparity = disparity.squeeze(0)  # remove channel
    disp_pixels = np.argwhere(disparity > 0)
    y, x = disp_pixels[:, 0], disp_pixels[:, 1]
    disp_valid = disparity[y, x]
    min_disp = disp_valid.min() if len(disp_valid) > 0 else 0
    max_disp = disp_valid.max() if len(disp_valid) > 0 else 0

    # disparity colormap (in reverse if depth map)
    norm = Normalize(vmin=min_disp, vmax=max_disp, clip=True)
    if mapper is None:
        mapper = cm.ScalarMappable(norm=norm, cmap="inferno")
    disp_color = mapper.to_rgba(disp_valid)[..., :3]
    image = np.zeros((*disparity.shape, 3))

    # to rgb ints
    image[y, x] = disp_color
    image = (image * 255).astype(np.uint8)

    return image, mapper


In [ ]:
from collections import OrderedDict
sequences = OrderedDict({
    "interlaken_00_a(540)": ("il0a", 540, 0.9),
    # "interlaken_00_b(500)": ("il0b", 500),
    "interlaken_00_b(560)": ("il0b", 560, 0.9),
    "interlaken_01_a(1680)": ("il1a", 1680, 0.9),
    "thun_01_b(400)": ("t1b", 400, 0.95),
    "zurich_city_12_a(440)": ("zc12a", 440, 0.9),
    "zurich_city_13_a(260)": ("zc13a", 260, 0.95),
})
sequences2 = OrderedDict({
    "interlaken_00_a(460)": ("il0a", 460, 0.9),
    "interlaken_00_a(540)": ("il0a", 540, 0.9),
    "interlaken_00_a(760)": ("il0a", 760, 0.9),
    "interlaken_00_b(500)": ("il0b", 500, 0.9),
    "interlaken_00_b(620)": ("il0b", 620, 0.9),
    "interlaken_00_b(740)": ("il0b", 740, 0.9),
})
sequences3 = OrderedDict({
    "interlaken_01_a(700)": ("il1a", 700, 0.95),
    # "interlaken_01_a(1080)": ("il1a", 1080),
    "interlaken_01_a(1980)": ("il1a", 1980, 0.9),
    "thun_01_a(120)": ("t1a", 120, 0.9),
    "zurich_city_12_a(280)": ("zc12a", 280, 0.95),
    "zurich_city_14_a(100)": ("zc14a", 100, 0.9),
    "zurich_city_15_a(540)": ("zc15a", 540, 0.9),
})

In [ ]:
seq = DsecSequence("../data/raw/dsec_old", "thun_01_b", 10000, rectify=True, gt="eval_disparity")

In [ ]:
seq[260//20]['frames'][0].max()

In [ ]:
plt.imshow(event_frame_to_rgb(seq[400//20]['frames'][0], saturation_percentile=0.95, gain=1.0))

In [ ]:
fig, axs = plt.subplots(len(sequences), 4, figsize=(15, 25), constrained_layout=True)
titles = ["Image", "Events", "Cho et al. ", "Ours"]
for i, (name, (sequence, frame, percentile)) in enumerate(sequences.items()):
    # print(name.split('(')[0])
    seq = DsecSequence("../data/raw/dsec_old", name.split('(')[0], 10000, rectify=True, gt="eval_disparity")
    for j in range(4):
        ax = axs[i, j]
        if j == 0:
            ax.set_ylabel(name, fontsize=20, labelpad=15)
        if i == 0:
            ax.set_title(titles[j], fontsize=30, pad=15)
        ax.tick_params(
            axis='both',          # changes apply to the x-axis
            which='both',      # both major and minor ticks are affected
            left=False,      # ticks along the bottom edge are off
            top=False,
            bottom=False,  # ticks along the top edge are off
            labelbottom=False,
            labelleft=False)
        
        if j == 0:
            # image
            ax.imshow(plt.imread(f"dsec_qual/{sequence}_{str(frame).zfill(6)}i.png"))


        if j == 1:
            # event image
            ax.imshow(event_frame_to_rgb(seq[frame//20]['frames'][0], saturation_percentile=percentile))

        if j == 2:
            # flow
            disparity = imageio.imread(
                f"dsec_qual/{sequence}_{str(frame-1).zfill(6)}.png").astype(np.float32) / 256
            disparity_img, mapper = disparity_map_to_image(np.expand_dims(disparity, 0))
            ax.imshow(disparity_img)

        if j == 3:
            # flow
            disparity = imageio.imread(
                f"dsec_qual/{sequence}_{str(frame).zfill(6)}.png").astype(np.float32) / 256
            disparity_img, _ = disparity_map_to_image(np.expand_dims(disparity, 0), mapper)
            ax.imshow(disparity_img)

fig.savefig('dsec.pdf')

In [ ]:
# sequences = sequences3
fig, axs = plt.subplots(4, len(sequences), figsize=(
    25, 13), constrained_layout=True)
titles = ["Image", "Events", "Cho et al. ", "Ours"]
for i, (name, (sequence, frame, percentile)) in enumerate(sequences.items()):
    seq = DsecSequence("../data/raw/dsec_old", name.split('(')[0], 10000, rectify=True, gt="eval_disparity")
    for j in range(4):
        ax = axs[j, i]
        if j == 0:
            ax.set_title(name, fontsize=25, pad=15)
        if i == 0:
            ax.set_ylabel(titles[j], fontsize=30, labelpad=15)
        ax.tick_params(
            axis='both',          # changes apply to the x-axis
            which='both',      # both major and minor ticks are affected
            left=False,      # ticks along the bottom edge are off
            top=False,
            bottom=False,  # ticks along the top edge are off
            labelbottom=False,
            labelleft=False)

        if j == 0:
            # image
            ax.imshow(plt.imread(
                f"dsec_qual/{sequence}_{str(frame).zfill(6)}i.png"))

        if j == 1:
            # event image
            ax.imshow(event_frame_to_rgb(seq[frame//20]['frames'][0], saturation_percentile=percentile))

        if j == 2:
            # flow
            disparity = imageio.imread(
                f"dsec_qual/{sequence}_{str(frame-1).zfill(6)}.png").astype(np.float32) / 256
            disparity_img, mapper = disparity_map_to_image(
                np.expand_dims(disparity, 0))
            ax.imshow(disparity_img)

        if j == 3:
            # flow
            disparity = imageio.imread(
                f"dsec_qual/{sequence}_{str(frame).zfill(6)}.png").astype(np.float32) / 256
            disparity_img, _ = disparity_map_to_image(
                np.expand_dims(disparity, 0), mapper)
            ax.imshow(disparity_img)

fig.savefig('dsec-landscape.pdf')